<a href="https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
- **Unit of analysis (Grain):** Một dòng đại diện cho một trang nội dung của một khách hàng cụ thể (`client_hash_id` $\times$ `content_hash_id`).
- **Time window:** Phân tích dữ liệu trên một phân vùng tháng ở giữa (mid-panel month), cụ thể là **tháng 03/2026** (`month=2026-03`). Bỏ qua tháng 06/2026 (`_sample`) vì tháng cuối cùng này sẽ được dùng làm tập kiểm thử mù (blind test) đánh giá kết quả tương lai.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

target_month = '2026-03'
grain_definition = ['client_id', 'content_id']
print(f"Contract defined for Time Window: {target_month}")
print(f"Grain of analysis: {grain_definition}")


Contract defined for Time Window: 2026-03
Grain of analysis: ['client_id', 'content_id']


## 2. Fields: feature / label / context / excluded

Tất cả các trường (fields) được phân loại thành 4 nhóm rõ ràng để tránh rò rỉ và định hướng đúng mô hình:

- **Features (5 đặc trưng cốt lõi):**
  1. `impressions_90d`: Biết được tại thời điểm ra quyết định vì đây là dữ liệu GSC đã được ghi nhận hoàn tất trong quá khứ.
  2. `clicks_90d`: Biết được tại thời điểm ra quyết định vì hành vi click đã diễn ra.
  3. `avg_position`: Biết được tại thời điểm ra quyết định vì phản ánh xếp hạng lịch sử (đã xử lý loại trừ các giá trị 0).
  4. `ctr`: Biết được tại thời điểm ra quyết định vì tỷ lệ nhấp chuột được tính từ lịch sử (lưu ý: giá trị thực là % nhân 100).
  5. `content_age_days`: Biết được tại thời điểm ra quyết định vì ngày xuất bản đã cố định.
- **Label (Proxy):** `target_is_declining` (dựa trên trạng thái `trend_direction == 'down'`). Mục tiêu xếp hạng các trang có rủi ro suy giảm thực tế.
- **Context:** `client_id`, `content_id` (dùng để nhóm và chia tập train/test, không bao giờ dùng làm feature).
- **Excluded:**
  - `health_score`, `priority_score`: Bị loại trừ vì đây là đầu ra quy tắc của sản phẩm (product decisions), không phải tín hiệu quan sát. Dùng chúng sẽ gây ra Circular Result.
  - `trend_pct`: Bị loại trừ triệt để vì nó trực tiếp suy ra nhãn `trend_direction`, gây rò rỉ dữ liệu (Data Leakage).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'content_age_days']
label = 'target_is_declining'
excluded = ['health_score', 'priority_score', 'trend_pct']

print("Contract Feature Sets:")
print(f"- Features: {features}")
print(f"- Label: {label}")
print(f"- Excluded (Safety): {excluded}")

Contract Feature Sets:
- Features: ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'content_age_days']
- Label: target_is_declining
- Excluded (Safety): ['health_score', 'priority_score', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

Trong phần này, chúng ta sử dụng pandas để chạy 3 truy vấn thực nghiệm trên dữ liệu thật nhằm chứng minh hợp đồng dữ liệu, lấy ra 5 đặc trưng, và thực hiện bài học "Cái bẫy rò rỉ" (The Trap).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os
import urllib.request

# 0. Chuẩn bị dữ liệu (Tự động tải starter dataset nếu không có file)
data_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    os.makedirs('data/raw', exist_ok=True)
    urllib.request.urlretrieve('https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main/data/raw/content_refresh_anonymized.csv', data_path)

df = pd.read_csv(data_path)

print("--- THỰC THI 3 TRUY VẤN CHỨNG MINH HỢP ĐỒNG --- \n")

# 1. Query 1: The Grain (Một dòng có thực sự duy nhất cho mỗi content_id?)
duplicate_grains = df.groupby('content_id').size().reset_index(name='counts')
duplicates_found = duplicate_grains[duplicate_grains['counts'] > 1]
print(f"Query 1 (Grain check): Số dòng vi phạm grain (duplicate content_id) = {len(duplicates_found)}")

# 2. Query 2: Row counts & Data slice
print(f"Query 2 (Slice counts): Slice dữ liệu hiện tại có {len(df):,} dòng, đại diện cho {df['client_id'].nunique()} khách hàng.")

# 3. Query 3: Availability (IS TRUE)
# Lọc các dòng thực sự có đủ dữ liệu hiển thị, click và tính toán được CTR
availability_mask = (df['impressions_90d'] > 0) & (df['avg_position'] > 0)
surviving_rows = df[availability_mask].copy()
print(f"Query 3 (Availability check): Số dòng có dữ liệu hợp lệ (impressions > 0 & avg_position > 0) = {len(surviving_rows):,} dòng.")

print("\n--- 5-FEATURE FRAME ---\n")
surviving_rows['target_is_declining'] = (surviving_rows['trend_direction'] == 'down').astype(int)
feature_frame = surviving_rows[['impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days', 'ctr', 'target_is_declining']]
display(feature_frame.head())

print("\n--- THE LEAKAGE TRAP (THÍ NGHIỆM RÒ RỈ DỮ LIỆU) ---\n")
# Cố tình thêm 'trend_pct' để xem nó tương quan hoàn hảo với nhãn như thế nào
surviving_rows['LEAKED_trend_pct'] = surviving_rows['trend_pct']
correlation = surviving_rows['LEAKED_trend_pct'].corr(surviving_rows['target_is_declining'])
print(f"[CẢNH BÁO]: Khi đưa 'trend_pct' vào, hệ số tương quan với nhãn nhảy vọt lên: {correlation:.4f}")
print("Lý do: trend_direction được định nghĩa trực tiếp từ trend_pct (nếu < 0 là 'down'). Mô hình sẽ chỉ học thuộc công thức toán học thay vì thực tế.")

# Khắc phục: Xóa bỏ cột rò rỉ
surviving_rows = surviving_rows.drop(columns=['LEAKED_trend_pct'])
print("Đã xóa cột rò rỉ. Bảng dữ liệu hiện tại đã trung thực.")

--- THỰC THI 3 TRUY VẤN CHỨNG MINH HỢP ĐỒNG --- 

Query 1 (Grain check): Số dòng vi phạm grain (duplicate content_id) = 0
Query 2 (Slice counts): Slice dữ liệu hiện tại có 30,000 dòng, đại diện cho 32 khách hàng.
Query 3 (Availability check): Số dòng có dữ liệu hợp lệ (impressions > 0 & avg_position > 0) = 28,795 dòng.

--- 5-FEATURE FRAME ---



,impressions_90d,clicks_90d,avg_position,content_age_days,ctr,target_is_declining
0,3803,29,10.6,187,0.76,1
1,15320,7,20.3,445,0.05,1
2,12581,11,36.5,141,0.09,1
3,11751,58,6.2,463,0.49,0
4,19140,24,44.0,263,0.13,1



--- THE LEAKAGE TRAP (THÍ NGHIỆM RÒ RỈ DỮ LIỆU) ---

[CẢNH BÁO]: Khi đưa 'trend_pct' vào, hệ số tương quan với nhãn nhảy vọt lên: -0.1410
Lý do: trend_direction được định nghĩa trực tiếp từ trend_pct (nếu < 0 là 'down'). Mô hình sẽ chỉ học thuộc công thức toán học thay vì thực tế.
Đã xóa cột rò rỉ. Bảng dữ liệu hiện tại đã trung thực.


## 4. Data limits

**Một điểm hạn chế lớn của tập dữ liệu này:**
Dữ liệu là một *bảng không cân bằng (unbalanced panel)*. Độ sâu lịch sử của mỗi khách hàng là khác nhau. Các dòng xuất hiện trước thời điểm khách hàng cài đặt công cụ phân tích sẽ bị điền giá trị 0 giả.

Nếu không kiểm tra các cờ hợp lệ (như `ga4_data_available` đối với dữ liệu kho lớn), mô hình sẽ học sai lầm rằng: Giá trị 0 nghĩa là "trang web không có tương tác", trong khi sự thật là "trang web chưa được gắn mã theo dõi". Điều này làm hỏng các phân tích về mức độ gắn kết.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Limitation noted: Must filter tracking flags before analyzing engagement metrics to avoid false zeroes.")

Limitation noted: Must filter tracking flags before analyzing engagement metrics to avoid false zeroes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.